# Contrail segmentation — fine-tune U-Net on GVCCS

**Dataset:** GVCCS (Ground Visible Camera Contrail Sequences)  
**Source:** Zenodo 15743988, CC BY 4.0, EUROCONTROL MUAC Brétigny-sur-Orge  
**Model:** U-Net + EfficientNet-B4 encoder (ImageNet pretrained)  
**Estimated time:** ~4–6 hours on Colab T4  

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells (Runtime → Run all)
3. Approve Google Drive access when prompted
4. Weights will be saved to `My Drive/coav_contrail_model/contrail_unet_best.pt`


In [ ]:
# ── 1. GPU check ──────────────────────────────────────────────────────────────
import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  |  VRAM: {vram:.1f} GB")
else:
    raise RuntimeError("No GPU found. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────────
!pip install segmentation-models-pytorch albumentations -q
print("Done.")

In [ ]:
# ── 3. Mount Google Drive (for saving checkpoints) ────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/coav_contrail_model'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints → {SAVE_DIR}")

In [ ]:
# ── 4. Download GVCCS dataset ─────────────────────────────────────────────────
#
# Option A (automatic): download from Zenodo  ~2.1 GB, ~5-10 min
# Option B (manual):    upload GVCCS.zip to Google Drive first, then:
#   !cp '/content/drive/MyDrive/GVCCS.zip' /content/GVCCS.zip
#
import os

if not os.path.exists('/content/GVCCS'):
    print("Downloading GVCCS from Zenodo (~2.1 GB)...")
    !wget -q --show-progress \
        'https://zenodo.org/records/15743988/files/GVCCS.zip?download=1' \
        -O /content/GVCCS.zip
    print("Extracting...")
    !unzip -q /content/GVCCS.zip -d /content/
    !rm /content/GVCCS.zip
    print("Done.")
else:
    print("GVCCS already present — skipping download.")

# Verify
import json
with open('/content/GVCCS/train/annotations.json') as f:
    _meta = json.load(f)
print(f"Images: {len(_meta['images'])}  |  Annotations: {len(_meta['annotations'])}")

In [ ]:
# ── 5. Dataset class ──────────────────────────────────────────────────────────
import json
import cv2
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset

class GVCCSDataset(Dataset):
    def __init__(self, data_dir, transform=None, indices=None):
        self.data_dir  = Path(data_dir)
        self.transform = transform

        with open(self.data_dir / 'annotations.json') as f:
            coco = json.load(f)

        self.ann_by_img = {}
        for ann in coco['annotations']:
            self.ann_by_img.setdefault(ann['image_id'], []).append(ann)

        imgs = coco['images']
        self.images = [imgs[i] for i in indices] if indices is not None else imgs

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        meta = self.images[idx]
        path = self.data_dir / 'images' / meta['file_name']

        image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)

        h = meta.get('height', 1024)
        w = meta.get('width',  1024)
        mask = np.zeros((h, w), dtype=np.uint8)
        for ann in self.ann_by_img.get(meta['id'], []):
            for seg in ann.get('segmentation', []):
                pts = np.array(seg, dtype=np.int32).reshape(-1, 2)
                cv2.fillPoly(mask, [pts], 1)

        if self.transform:
            r     = self.transform(image=image, mask=mask)
            image = r['image']
            mask  = r['mask'].unsqueeze(0).float()

        return image, mask

print("GVCCSDataset defined.")

In [ ]:
# ── 6. Transforms & DataLoaders ───────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

IMG_SIZE   = 512
BATCH_SIZE = 8   # fits T4 16 GB; reduce to 4 if OOM
DATA_DIR   = '/content/GVCCS/train'

# Count total images for split
with open(f'{DATA_DIR}/annotations.json') as f:
    _n = len(json.load(f)['images'])

val_size    = int(_n * 0.1)
train_idx   = list(range(_n - val_size))
val_idx     = list(range(_n - val_size, _n))

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussianBlur(blur_limit=3, p=0.15),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

train_ds = GVCCSDataset(DATA_DIR, transform=train_tf, indices=train_idx)
val_ds   = GVCCSDataset(DATA_DIR, transform=val_tf,   indices=val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} images ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds)} images ({len(val_loader)} batches)")

In [ ]:
# ── 7. Model, loss, optimizer ─────────────────────────────────────────────────
import segmentation_models_pytorch as smp
import torch

model = smp.Unet(
    encoder_name='efficientnet-b4',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,          # raw logits → we apply sigmoid in loss/metrics
).cuda()

# Dice + BCE — standard for thin-structure binary segmentation
dice_loss = smp.losses.DiceLoss(mode='binary')
bce_loss  = smp.losses.SoftBCEWithLogitsLoss()
loss_fn   = lambda p, t: dice_loss(p, t) + bce_loss(p, t)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: U-Net / EfficientNet-B4  |  Params: {total_params:.1f}M")

In [ ]:
# ── 8. Training loop ──────────────────────────────────────────────────────────
from tqdm.notebook import tqdm
import time

EPOCHS    = 30
best_dice = 0.0
history   = []

def dice_score(logits, target, threshold=0.5):
    pred  = (torch.sigmoid(logits) > threshold).float()
    inter = (pred * target).sum()
    return (2 * inter / (pred.sum() + target.sum() + 1e-8)).item()


for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── train ──
    model.train()
    tr_loss = tr_dice = 0.0
    for images, masks in tqdm(train_loader, desc=f'Ep {epoch:02d}/{EPOCHS} train', leave=False):
        images, masks = images.cuda(), masks.cuda()
        optimizer.zero_grad()
        logits = model(images)
        loss   = loss_fn(logits, masks)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()
        tr_dice += dice_score(logits, masks)

    # ── val ──
    model.eval()
    vl_loss = vl_dice = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f'Ep {epoch:02d}/{EPOCHS} val  ', leave=False):
            images, masks = images.cuda(), masks.cuda()
            logits = model(images)
            vl_loss += loss_fn(logits, masks).item()
            vl_dice += dice_score(logits, masks)

    scheduler.step()

    nb  = len(train_loader)
    nvb = len(val_loader)
    ep_dice = vl_dice / nvb
    elapsed = time.time() - t0

    row = dict(epoch=epoch,
               tr_loss=tr_loss/nb, tr_dice=tr_dice/nb,
               vl_loss=vl_loss/nvb, vl_dice=ep_dice)
    history.append(row)

    marker = ''
    if ep_dice > best_dice:
        best_dice = ep_dice
        torch.save(model.state_dict(), f'{SAVE_DIR}/contrail_unet_best.pt')
        marker = '  ← best'

    print(f"Ep {epoch:02d}/{EPOCHS}  "
          f"tr loss={row['tr_loss']:.4f} dice={row['tr_dice']:.4f}  "
          f"val loss={row['vl_loss']:.4f} dice={ep_dice:.4f}  "
          f"{elapsed:.0f}s{marker}")

print(f"\nTraining complete. Best val Dice: {best_dice:.4f}")
print(f"Weights: {SAVE_DIR}/contrail_unet_best.pt")

In [ ]:
# ── 9. Loss / Dice curves ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs = [r['epoch'] for r in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, [r['tr_loss'] for r in history], label='train')
ax1.plot(epochs, [r['vl_loss'] for r in history], label='val')
ax1.set_title('Loss (Dice + BCE)'); ax1.legend()

ax2.plot(epochs, [r['tr_dice'] for r in history], label='train')
ax2.plot(epochs, [r['vl_dice'] for r in history], label='val')
ax2.set_title('Dice score'); ax2.legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=120)
plt.show()

In [ ]:
# ── 10. Visual check on 4 val images ─────────────────────────────────────────
import matplotlib.pyplot as plt
import random

model.load_state_dict(torch.load(f'{SAVE_DIR}/contrail_unet_best.pt'))
model.eval()

with open(f'{DATA_DIR}/annotations.json') as f:
    _coco = json.load(f)
_all_imgs = _coco['images']

samples = random.sample(val_idx, 4)
fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for row, idx in enumerate(samples):
    meta  = _all_imgs[idx]
    img   = cv2.cvtColor(cv2.imread(f'{DATA_DIR}/images/{meta["file_name"]}'), cv2.COLOR_BGR2RGB)
    inp   = val_tf(image=img)['image'].unsqueeze(0).cuda()

    with torch.no_grad():
        pred = torch.sigmoid(model(inp))[0, 0].cpu().numpy()

    axes[row, 0].imshow(img);              axes[row, 0].set_title('Input')
    axes[row, 1].imshow(pred, cmap='hot'); axes[row, 1].set_title('Heatmap')
    axes[row, 2].imshow(pred > 0.5, cmap='gray'); axes[row, 2].set_title('Mask (t=0.5)')
    for ax in axes[row]: ax.axis('off')

plt.suptitle(f'Best val Dice = {best_dice:.4f}', fontsize=14)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/sample_predictions.png', dpi=120)
plt.show()
print(f"Sample predictions saved → {SAVE_DIR}/sample_predictions.png")

In [ ]:
# ── 11. What to do next ───────────────────────────────────────────────────────
print(f"""
=== Done ===

Saved to Google Drive:
  {SAVE_DIR}/contrail_unet_best.pt   ← weights
  {SAVE_DIR}/training_curves.png
  {SAVE_DIR}/sample_predictions.png

To use in the edge-pi project:
  1. Download contrail_unet_best.pt from Google Drive
  2. Copy to:  edge-pi/python/weights/contrail_unet.pt
  3. python inference.py --image your_image.jpg
     (inference.py auto-detects .pt weights and uses PyTorch backend)

Interpretation:
  Dice > 0.60  → usable for demo
  Dice > 0.75  → good for production PoC
  Dice < 0.40  → try more epochs or larger encoder (efficientnet-b6)
""")